# NOAA Daily Weather Data

## Purpose

Build a daily **weather feature table** for each of the 10 M5 stores. The M5 dataset contains no weather information, yet weather directly affects retail foot traffic — rain reduces store visits, blizzards can close stores, and extreme heat shifts category demand. The key insight: **deviation from normal** matters more than absolute temperature. A 50°F day in CA is cold; in WI it's pleasant.

### Data Source
- NOAA Climate Data Online (CDO) API — GHCND dataset
- 10 airport weather stations mapped to 10 M5 stores

In [ ]:
# ── Uncomment below when running on Google Colab ────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.chdir('/content/drive/MyDrive/Colab Notebooks/m5-forecasting-accuracy/notebooks')

BASE_DIR = '..'
RAW_DIR = f'{BASE_DIR}/data/raw'
CLEAN_DIR = f'{BASE_DIR}/data/clean'
FIG_DIR = f'{BASE_DIR}/figures'

import os
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
print(f'Project dir: {BASE_DIR}')

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import os
import warnings
warnings.filterwarnings('ignore')

# RAW_DIR defined in mount cell above
# CLEAN_DIR defined in mount cell above

DATE_START = '2011-01-29'
DATE_END = '2016-06-19'

try:
    from google.colab import userdata
    NOAA_TOKEN = userdata.get('NOAA_TOKEN')
except:
    NOAA_TOKEN = os.environ.get('NOAA_TOKEN', '')
assert NOAA_TOKEN, 'Set NOAA_TOKEN in Colab Secrets (key icon in sidebar) or as env var'
NOAA_BASE = 'https://www.ncdc.noaa.gov/cdo-web/api/v2'
print(f'NOAA API token loaded ({NOAA_TOKEN[:8]}...)')

## Step 1: Store-to-station mapping

We map each M5 store to the nearest major-airport NOAA weather station. Airport stations have >99% data completeness for 2011-2016.

**Why store-level?** LA (avg 75F) and San Francisco (avg 60F) have very different climates within the same state.

In [ ]:
STORE_STATION_MAP = {
    'CA_1': {'station': 'GHCND:USW00023174', 'name': 'Los Angeles Intl Airport', 'state': 'CA'},
    'CA_2': {'station': 'GHCND:USW00093193', 'name': 'Fresno Yosemite Intl', 'state': 'CA'},
    'CA_3': {'station': 'GHCND:USW00023232', 'name': 'Sacramento Executive', 'state': 'CA'},
    'CA_4': {'station': 'GHCND:USW00023234', 'name': 'San Francisco Intl', 'state': 'CA'},
    'TX_1': {'station': 'GHCND:USW00013960', 'name': 'Dallas/Fort Worth Intl', 'state': 'TX'},
    'TX_2': {'station': 'GHCND:USW00012918', 'name': 'Houston Hobby', 'state': 'TX'},
    'TX_3': {'station': 'GHCND:USW00012921', 'name': 'San Antonio Intl', 'state': 'TX'},
    'WI_1': {'station': 'GHCND:USW00014839', 'name': 'Milwaukee Mitchell', 'state': 'WI'},
    'WI_2': {'station': 'GHCND:USW00014837', 'name': 'Madison Dane County', 'state': 'WI'},
    'WI_3': {'station': 'GHCND:USW00014898', 'name': 'Green Bay Austin Straubel', 'state': 'WI'},
}
for s, i in STORE_STATION_MAP.items():
    print(f'  {s} -> {i["name"]}')

  CA_1 -> Los Angeles Intl Airport
  CA_2 -> Fresno Yosemite Intl
  CA_3 -> Sacramento Executive
  CA_4 -> San Francisco Intl
  TX_1 -> Dallas/Fort Worth Intl
  TX_2 -> Houston Hobby
  TX_3 -> San Antonio Intl
  WI_1 -> Milwaukee Mitchell
  WI_2 -> Madison Dane County
  WI_3 -> Green Bay Austin Straubel


## Step 2: NOAA API fetch function

The API has constraints: max 1 year per query, max 1000 records per response, rate limit ~5 req/sec. We handle all three with yearly chunking, pagination, and retries with increasing timeouts.

In [ ]:
DATATYPES = ['TMAX', 'TMIN', 'PRCP', 'SNOW']

def fetch_noaa_data(station_id, start_date, end_date, token):
    headers = {'token': token}
    all_results = []
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    current_start = start
    while current_start < end:
        current_end = min(current_start + pd.DateOffset(years=1) - pd.DateOffset(days=1), end)
        offset = 1
        while True:
            params = {
                'datasetid': 'GHCND', 'stationid': station_id,
                'datatypeid': ','.join(DATATYPES),
                'startdate': current_start.strftime('%Y-%m-%d'),
                'enddate': current_end.strftime('%Y-%m-%d'),
                'units': 'standard', 'limit': 1000, 'offset': offset,
            }
            resp = None
            for attempt, t in enumerate([60, 90, 120]):
                try:
                    resp = requests.get(f'{NOAA_BASE}/data', headers=headers, params=params, timeout=t)
                    break
                except requests.exceptions.ReadTimeout:
                    print(f'    Timeout (attempt {attempt+1}), retrying...')
                    time.sleep(3)
            if resp is None:
                print('    All retries failed, skipping chunk')
                break
            if resp.status_code == 429:
                print('    Rate limited, waiting 5s...')
                time.sleep(5)
                continue
            if resp.status_code != 200:
                print(f'    HTTP {resp.status_code}, skipping chunk')
                break
            data = resp.json()
            results = data.get('results', [])
            if not results:
                break
            all_results.extend(results)
            total_count = data.get('metadata', {}).get('resultset', {}).get('count', 0)
            if offset + 1000 > total_count:
                break
            offset += 1000
            time.sleep(0.3)
        current_start = current_end + pd.DateOffset(days=1)
        time.sleep(0.5)
    return all_results

print('fetch_noaa_data() defined — retries up to 3x with 60/90/120s timeouts')

fetch_noaa_data() defined — retries up to 3x with 60/90/120s timeouts


## Step 3: Fetch all 10 stations

This will take a few minutes — 10 stations x ~6 yearly chunks = ~60 API calls.

In [ ]:
all_weather = []
for store_id, info in STORE_STATION_MAP.items():
    print(f'Fetching {store_id} ({info["name"]})...', end=' ')
    results = fetch_noaa_data(info['station'], DATE_START, DATE_END, NOAA_TOKEN)
    print(f'{len(results)} records')
    for r in results:
        r['store_id'] = store_id
        r['state_id'] = info['state']
    all_weather.extend(results)
    time.sleep(1)

weather_raw = pd.DataFrame(all_weather)
print(f'\nTotal raw records: {len(weather_raw)}')
weather_raw.head()

Fetching CA_1 (Los Angeles Intl Airport)...     HTTP 503, skipping chunk
5812 records
Fetching CA_2 (Fresno Yosemite Intl)...     HTTP 503, skipping chunk
7416 records
Fetching CA_3 (Sacramento Executive)... 5968 records
Fetching CA_4 (San Francisco Intl)...     HTTP 503, skipping chunk
4904 records
Fetching TX_1 (Dallas/Fort Worth Intl)... 7876 records
Fetching TX_2 (Houston Hobby)...     HTTP 503, skipping chunk
5871 records
Fetching TX_3 (San Antonio Intl)... 7876 records
Fetching WI_1 (Milwaukee Mitchell)... 7876 records
Fetching WI_2 (Madison Dane County)... 7876 records
Fetching WI_3 (Green Bay Austin Straubel)...     HTTP 503, skipping chunk
7416 records

Total raw records: 68891


,date,datatype,station,attributes,value,store_id,state_id
0,2011-01-29T00:00:00,PRCP,GHCND:USW00023174,",,X,2400",0.00,CA_1,CA
1,2011-01-29T00:00:00,TMAX,GHCND:USW00023174,",,X,2400",62.00,CA_1,CA
2,2011-01-29T00:00:00,TMIN,GHCND:USW00023174,",,X,2400",47.00,CA_1,CA
3,2011-01-30T00:00:00,PRCP,GHCND:USW00023174,",,X,2400",0.16,CA_1,CA
4,2011-01-30T00:00:00,TMAX,GHCND:USW00023174,",,X,2400",59.00,CA_1,CA


## Step 4: Pivot and clean

NOAA returns long format (one row per datatype per day). We pivot to wide: one row per `(store, date)` with TMAX/TMIN/PRCP/SNOW as columns.

In [ ]:
weather_raw['date'] = pd.to_datetime(weather_raw['date'].str[:10])
weather = weather_raw.pivot_table(
    index=['store_id', 'state_id', 'date'], columns='datatype',
    values='value', aggfunc='first'
).reset_index()
weather = weather.rename(columns={
    'TMAX': 'temp_high_f', 'TMIN': 'temp_low_f',
    'PRCP': 'precipitation_in', 'SNOW': 'snowfall_in',
})
weather['precipitation_in'] = weather['precipitation_in'].fillna(0)
weather['snowfall_in'] = weather['snowfall_in'].fillna(0)
weather = weather.sort_values(['store_id', 'date'])
weather['temp_high_f'] = weather.groupby('store_id')['temp_high_f'].ffill().bfill()
weather['temp_low_f'] = weather.groupby('store_id')['temp_low_f'].ffill().bfill()
print(f'Pivoted shape: {weather.shape}')
print(f'Missing values:\n{weather.isnull().sum()}')

Pivoted shape: (19001, 7)
Missing values:
datatype
store_id            0
state_id            0
date                0
precipitation_in    0
snowfall_in         0
temp_high_f         0
temp_low_f          0
dtype: int64


## Step 5: Feature engineering

| Feature | Why |
|---|---|
| `temp_range` | Large swings = unstable weather |
| `temp_deviation` | **Key feature**: degrees above/below 30-day rolling mean |
| `is_severe_weather` | Heavy precip (>1 in) OR extreme temp anomaly (>2 SD) |
| `rainy_weekend` | Rain + weekend interaction — discretionary trips get cancelled |

In [ ]:
weather['temp_range'] = weather['temp_high_f'] - weather['temp_low_f']
weather['is_rainy'] = (weather['precipitation_in'] > 0.1).astype(int)
weather['is_snowy'] = (weather['snowfall_in'] > 0).astype(int)
weather['temp_rolling_mean_30d'] = weather.groupby('store_id')['temp_high_f'].transform(
    lambda x: x.rolling(30, min_periods=7).mean())
weather['temp_deviation'] = weather['temp_high_f'] - weather['temp_rolling_mean_30d']
weather['temp_deviation'] = weather['temp_deviation'].fillna(0).astype('float32')  # First 30 days have no rolling mean
temp_std_30d = weather.groupby('store_id')['temp_high_f'].transform(
    lambda x: x.rolling(30, min_periods=7).std())
weather['is_severe_weather'] = (
    (weather['precipitation_in'] > 1.0) |
    (weather['temp_deviation'].abs() > 2 * temp_std_30d)
).astype(int)
weather['day_of_week'] = weather['date'].dt.dayofweek
weather['rainy_weekend'] = ((weather['day_of_week'] >= 5) & (weather['is_rainy'] == 1)).astype(int)
weather = weather.drop(columns=['temp_rolling_mean_30d', 'day_of_week'])
print(f'Weather features shape: {weather.shape}')
print(f'Severe weather days: {weather["is_severe_weather"].sum()} ({100*weather["is_severe_weather"].mean():.1f}%)')
print(f'Rainy days: {weather["is_rainy"].sum()} ({100*weather["is_rainy"].mean():.1f}%)')
print(f'Snowy days: {weather["is_snowy"].sum()} ({100*weather["is_snowy"].mean():.1f}%)')

Weather features shape: (19001, 13)
Severe weather days: 1550 (8.2%)
Rainy days: 2346 (12.3%)
Snowy days: 589 (3.1%)


### Interpretation

**8.2% of store-days are classified as severe weather** (heavy precipitation >1 inch or temperature anomaly exceeding 2 standard deviations from the 30-day mean). This is frequent enough to provide meaningful signal without being so rare that the model cannot learn from it.

Rain affects **12.3%** of store-days, while snowfall is concentrated in Wisconsin at **3.1%** overall. The `temp_deviation` feature is particularly important: it captures whether a given day's temperature is unusual *for that store at that time of year*, which matters more for shopping behavior than the absolute temperature itself.

## Step 6: Sanity checks

In [ ]:
print('Average high temperature by state:')
for state in ['CA', 'TX', 'WI']:
    avg = weather[weather['state_id'] == state]['temp_high_f'].mean()
    print(f'  {state}: {avg:.1f}F')
print(f'\nSnow days by state:')
for state in ['CA', 'TX', 'WI']:
    snow = weather[weather['state_id'] == state]['is_snowy'].sum()
    total = len(weather[weather['state_id'] == state])
    print(f'  {state}: {snow} / {total} ({100*snow/total:.1f}%)')

Average high temperature by state:
  CA: 73.4F
  TX: 80.2F
  WI: 56.5F

Snow days by state:
  CA: 0 / 7334 (0.0%)
  TX: 16 / 5875 (0.3%)
  WI: 573 / 5792 (9.9%)


### Interpretation

All sanity checks pass. **CA averages 73.4°F**, **TX 80.2°F**, and **WI 56.5°F**, consistent with known climate differences. Snow is absent in California (0.0%), negligible in Texas (0.3%), and meaningful in Wisconsin (9.9%) — validating that our station-to-store mapping captures real weather variation across the three states.

## Step 7: Save output

**Merge key**: `(store_id, date)` — store-level because weather varies within a state.

In [ ]:
weather.to_csv(f'{CLEAN_DIR}/weather_daily.csv', index=False)
print(f'Saved: weather_daily.csv ({len(weather)} rows)')
print(f'Columns: {list(weather.columns)}')
print(f'Stores: {sorted(weather["store_id"].unique())}')
print(f'Date range: {weather["date"].min()} to {weather["date"].max()}')
print(f'Merge key for downstream: (store_id, date)')

Saved: weather_daily.csv (19001 rows)
Columns: ['store_id', 'state_id', 'date', 'precipitation_in', 'snowfall_in', 'temp_high_f', 'temp_low_f', 'temp_range', 'is_rainy', 'is_snowy', 'temp_deviation', 'is_severe_weather', 'rainy_weekend']
Stores: ['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2', 'WI_3']
Date range: 2011-01-29 00:00:00 to 2016-06-19 00:00:00
Merge key for downstream: (store_id, date)


---
## Conclusion

This notebook produces `weather_daily.csv` — daily high temperature, precipitation, and derived features (`temp_deviation`, `is_severe_weather`, `is_rainy`) for each of the 10 M5 stores via NOAA CDO API.

**Known limitation**: NOAA's API returns intermittent 503 errors for some station-year combinations, leaving gaps that are filled with `ffill().bfill()`. This silently propagates last-known values across multi-day gaps. For a more rigorous build, we would (a) audit gap sizes per station, (b) document any gap exceeding 7 days, and (c) consider falling back to a neighboring station rather than forward-filling. Current pipeline accepts this as a known data-quality cost.

**Merge key for downstream**: `(store_id, date)`.